# Comprehensive Study Notes

## Prompt Engineering Techniques & Retrieval-Augmented Generation (RAG)

*(Mentoring Session – ~2.5 hours)*

---

## 1. Conceptual Shift: From Prompt Engineering to Context Engineering

### 1.1 Why Prompt Engineering Alone Is No Longer Enough

Early LLM usage focused on:

* Clever phrasing
* Prompt templates
* Zero-shot / few-shot patterns
* Chain-of-Thought prompting

While these techniques remain valid, the session emphasized that **they break down at scale** because:

* Prompts become brittle when models change
* Larger systems depend more on *what context is provided* than *how the question is worded*
* Tool use, retrieval, and agents introduce multiple prompt entry points

**Key Insight:**

> Modern LLM systems fail less because of “bad prompts” and more because of **poor context selection, filtering, and evaluation**.

---

### 1.2 Context Engineering Defined

Context engineering is the discipline of:

* Deciding **what information enters the model**
* Deciding **when it enters**
* Deciding **how much enters**
* Deciding **what format it enters in**

This includes:

* Retrieved documents
* Tool outputs
* Intermediate agent reasoning
* Metadata, scores, and constraints

Prompt engineering becomes just **one layer** inside a broader context pipeline.

---

## 2. Prompt Engineering as a Control Mechanism (Not a Trick)

### 2.1 System Prompts as the “Policy Layer”

The session repeatedly emphasized the **system prompt** as the highest-leverage prompt:

System prompts are used to:

* Enforce behavior boundaries
* Define evaluation criteria
* Specify output schemas
* Control reasoning style
* Prevent hallucination

Rather than being verbose, effective system prompts are:

* Explicit
* Structured
* Deterministic

They act as a **contract** between the system and the model.

---

### 2.2 Structured Prompts and Structured Outputs

A strong recommendation from the session:

> **If you can structure it, structure it.**

Structured outputs:

* JSON
* Typed fields
* Boolean flags
* Confidence scores

Benefits:

* Deterministic parsing
* Easier evaluation
* Model-agnostic behavior
* Reduced ambiguity

This concept appeared in multiple forms:

* “Signatures” (DSPy)
* Pydantic models (LangChain)
* Typed scorers and graders

**Important takeaway:**
Free-form text is a liability in production systems.

---

### 2.3 Prompting for Scoring, Not Just Answers

The session went beyond “answer generation” and focused on **LLMs as evaluators**.

Examples:

* Is this document relevant? (Boolean)
* How confident is this relevance? (0–1)
* Why was it considered relevant? (Reasoning)

This enables:

* Ranking
* Filtering
* Feedback loops
* Observability

---

## 3. RAG Ingestion Pipeline (Index-Time Concerns)

A major portion of the session focused on **what happens before the user ever asks a question**.

### 3.1 Data Loading

RAG systems can ingest:

* PDFs
* Text files
* Structured data
* Tables
* Logs

The key is **normalization**:

* Convert everything into text or structured representations
* Preserve metadata (source, page, section)

---

### 3.2 Chunking Is a Design Decision, Not a Default

Chunking was treated as **one of the most critical RAG decisions**.

Key ideas:

* Chunk size affects retrieval recall
* Overlap prevents loss of semantic continuity
* Poor chunking leads to:

  * Partial answers
  * Missed context
  * Irrelevant retrieval

Two strategies:

1. **Structure-based chunking** (preferred)
2. **Length-based chunking** with overlap (fallback)

Overlap exists to **preserve meaning across boundaries**, not as an optimization hack.

---

### 3.3 Embeddings and Semantic Representation

Embeddings:

* Convert text into numerical vectors
* Enable semantic similarity search

Key points:

* Vector stores contain **numbers only**, not meaning
* Retrieval quality depends heavily on embedding choice
* Different embeddings serve different purposes:

  * Semantic similarity
  * Classification
  * Clustering
  * Multimodal alignment

Embedding choice is a **modeling decision**, not a configuration detail.

---

## 4. Retrieval-Time Intelligence

### 4.1 Initial Retrieval Is Often Noisy

Vector similarity search retrieves candidates, not truth.

Typical pattern:

* Retrieve top 50–100 chunks
* Expect many to be weakly relevant

This is normal and expected.

---

### 4.2 Re-Ranking as a Second Brain

Re-ranking models:

* Take the query + retrieved chunks
* Score true relevance
* Reorder results

This step:

* Dramatically improves grounding
* Reduces hallucination
* Limits prompt bloat

Only the **top-K re-ranked chunks** are sent to the LLM.

---

## 5. Query Understanding and Reformulation

### 5.1 Users Ask Bad Questions (That’s Normal)

User questions are often:

* Vague
* Contextless
* Ambiguous
* Overloaded

Instead of retrieving directly:

* The system **rephrases the question**
* Clarifies intent
* Narrows scope

This happens:

* As a preprocessing step
* Or inside a feedback loop

---

### 5.2 Why Paraphrasing Improves Retrieval

Retrievers respond to **semantic clarity**, not intent.

Rewriting the query:

* Aligns with stored document language
* Improves embedding similarity
* Increases recall and precision

This is a core **agentic RAG technique**.

---

## 6. Agentic RAG Architecture (End-to-End)

The session described RAG as a **multi-stage agentic system**, not a pipeline:

### Typical Flow:

1. User question
2. Query rewriting
3. Initial retrieval
4. Re-ranking
5. Document grading
6. Context selection
7. Prompt construction
8. Generation
9. Evaluation
10. Feedback loop (optional)

Each step can:

* Fail independently
* Be evaluated independently
* Be improved independently

---

## 7. Evaluation as a First-Class Citizen

### 7.1 Retrieval Evaluation (Before Generation)

Documents are evaluated for:

* Relevance
* Coverage
* Noise

This prevents:

* Garbage-in → garbage-out
* Hallucinations caused by weak context

---

### 7.2 Generation Evaluation (After Generation)

Three core metrics dominated the discussion:

1. **Groundedness / Faithfulness**

   * Answer must come from provided context
2. **Context Relevance**

   * Retrieved context must match the question
3. **Answer Relevance**

   * Answer must address the question directly

These metrics can be:

* Human-evaluated
* LLM-evaluated
* Used in feedback loops

---

### 7.3 Using Classification Metrics

When golden datasets exist:

* Precision
* Recall
* Accuracy
* F1

These are especially useful for:

* Retrieval validation
* Regression testing
* System benchmarking

---

## 8. Feedback Loops and Failure Handling

### 8.1 Iterative Improvement

If metrics are low:

* Retrieve more documents
* Re-rank again
* Rephrase the query

This can repeat multiple times.

---

### 8.2 Knowing When to Stop

A critical production insight:

> Sometimes the correct answer is **“I don’t have enough context.”**

After several failed attempts:

* System should fail gracefully
* Avoid hallucination at all costs

---

## 9. Prompt Lifecycle, Versioning, and Model Drift

### 9.1 Prompt Iteration Is Hard

Challenges discussed:

* Versioning prompts
* A/B testing
* Regressions
* Hidden coupling with models

Prompt engineering is **not static**.

---

### 9.2 Model Portability Is a Myth Without Structure

Switching models:

* Open-source → proprietary
* Vendor A → Vendor B

Breaks:

* Prompt assumptions
* Reasoning behavior
* Output consistency

Structured prompts + structured outputs mitigate this risk.

---

## 10. Strategic Takeaways from the Session

* Prompt engineering is necessary but insufficient
* Context engineering is the dominant skill
* Retrieval quality matters more than clever prompts
* Re-ranking and evaluation are essential
* Structured outputs are non-negotiable in production
* Agentic RAG systems are iterative, evaluative, and defensive
* The goal is **faithfulness, not fluency**



In [ ]:
import os

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain

In [ ]:
file_name = 'config.json'
with open(file_name, 'r') as file:
    config = json.load(file)
    os.environ['OPENAI_API_KEY'] = config.get("API_KEY") # Loading the API Key
    os.environ["OPENAI_BASE_URL"] = config.get("OPENAI_API_BASE") # Loading the API Base Url

In [ ]:
PDF_PATH = "your_document.pdf"
PERSIST_DIR = "./chroma_db"   # persists vectors to disk
TOP_K = 5


In [ ]:
def build_vectorstore(pdf_path: str) -> Chroma:
    # 1) Load
    docs = PyPDFLoader(pdf_path).load()

    # 2) Chunk (size + overlap like your session: ~512 + overlap)
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=512,
        chunk_overlap=16,
    )
    chunks = splitter.split_documents(docs)

    # 3) Embed + store (persisted)
    embeddings = OpenAIEmbeddings()
    vs = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=PERSIST_DIR,
        collection_name="rag_notes_demo",
    )
    return vs

In [ ]:
def build_rag_chain(vectorstore: Chroma):
    # Retriever (top-k)
    retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})

    # Deterministic model for RAG
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    # “Answer strictly from context; if missing say I don’t know.”
    prompt = ChatPromptTemplate.from_messages([
        ("system",
         "You are a Q&A assistant. Use ONLY the provided context to answer.\n"
         "If the answer is not in the context, say: \"I don't know.\""),
        ("human",
         "CONTEXT:\n{context}\n\nQUESTION:\n{input}")
    ])

    # Stuff docs into {context} (simple + effective baseline)
    doc_chain = create_stuff_documents_chain(llm, prompt)

    # Retrieval chain: retrieve -> stuff -> generate
    rag_chain = create_retrieval_chain(retriever, doc_chain)
    return rag_chain

In [ ]:
if not os.path.exists(PERSIST_DIR):
    vectorstore = build_vectorstore(PDF_PATH)
else:
    vectorstore = Chroma(
        persist_directory=PERSIST_DIR,
        embedding_function=OpenAIEmbeddings(),
        collection_name="rag_notes_demo",
    )

rag_chain = build_rag_chain(vectorstore)

# Ask a question
question = "Who are the authors and where was this published?"
result = rag_chain.invoke({"input": question})

print("\nAnswer:\n", result["answer"])

# Optional: inspect retrieved chunks
print("\n--- Retrieved Chunks (metadata) ---")
for i, d in enumerate(result["context"], start=1):
    print(f"{i}. source={d.metadata.get('source')} page={d.metadata.get('page')}")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

judge_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are evaluating if an answer is grounded in the provided context.\n"
     "Score 1-5:\n"
     "1 = not grounded / contradicts context\n"
     "3 = partially grounded\n"
     "5 = fully grounded\n"
     "Return JSON only: {\"score\": <1-5>, \"rationale\": \"...\"}"),
    ("human",
     "QUESTION:\n{question}\n\nCONTEXT:\n{context}\n\nANSWER:\n{answer}")
])

def judge_groundedness(question: str, context_docs, answer: str):
    context_text = "\n\n".join([d.page_content for d in context_docs])
    msg = judge_prompt.format_messages(
        question=question,
        context=context_text,
        answer=answer
    )
    return judge_llm.invoke(msg).content
